# Stage 3 - Contig generation from a solved path

Generic code for turning a solved Hamiltonian path into a contig, and for preparing that contig
for organism identification by BLASTn.

The procedure is the same for **every** subgraph reported in this study; only the input files and
the solved path change. It is demonstrated here on the **24-node African swine fever virus graph**
(SRA `SRR27477754`), the worked example used throughout.

**Inputs** (both carried over from the earlier stages)

- `pairwise_alignment.txt` - one record per scored read pair:
  `('<aligned sequence>', <score>, <read i>, <read j>)`. Used to rebuild the graph so a solved
  path can be scored and its original read ids recovered.
- `cleaned_reads.txt` - the retained reads from `1_multithread_alignment.py`, one sequence per
  line, so line number = read id. These are the sequences the contig is built from.

**Output.** The assembled contig, plus the BLASTn query used to identify the source organism.

**Pipeline position.** reads -> duplicate removal -> overlap scoring -> subgraph + Hamiltonian
verification (Stage 2) -> solver -> **this notebook**.


In [2]:
# Setup: imports and the graph drawing helper.
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from Bio import pairwise2


def show_wgraph(G):
    """Draw a weighted directed graph with its edge weights labelled."""
    plt.figure()
    pos = nx.spring_layout(G)
    weight_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw(G, pos, font_color='white', font_size=6, node_shape='s',
            node_size=70, with_labels=True)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=weight_labels, font_size=6)

In [3]:
# Read the pairwise alignment records. Each record is
# ('<aligned sequence>', score, read_i, read_j); the score becomes the weight of the
# directed edge read_i -> read_j. raw_records keeps the full lines so the alignment
# behind any particular edge can be printed later.
ALIGNMENT_FILE = "pairwise_alignment.txt"

scores = []
edge_source = []
edge_target = []
raw_records = []

with open(ALIGNMENT_FILE, 'r') as file:
    for line in file:
        raw_records.append(line.strip())
        columns = line.split(',')
        scores.append(columns[1])
        edge_source.append(columns[2])
        edge_target.append(columns[3].split(')')[0].strip())

scores = np.array(scores, dtype=float)
edge_source = np.array(edge_source, dtype=float)
edge_target = np.array(edge_target, dtype=float)
print("alignment records read:", len(scores))

alignment records read: 276


In [4]:
# The node order returned by the graph-solving stage, for this graph: the best of the
# 24-node hardware runs (Table II). solver_edge_weights holds the overlap weight of each
# consecutive pair, taken from the graph.
#
# The four 0.0 entries are the four VIOLATIONS reported for this run - consecutive nodes
# that are not joined by an edge of the graph, so there is no overlap to collapse. Their
# total, 269, is below the classical reference of 335, and at those four junctions contig
# generation has to append a whole read; that is how violations become reduced BLASTn
# query coverage (75% here against 78% classically, Table III).
solver_path = [11, 21, 23, 7, 9, 8, 19, 20, 12, 13, 17, 3, 4, 10, 0, 2, 22, 18, 14, 6, 16, 5, 15, 1]
solver_edge_weights = [13.0, 17.0, 16.0, 14.0, 0.0, 13.0, 21.0, 14.0, 13.0, 0.0, 13.0, 12.0,
                       9.0, 0.0, 16.0, 14.0, 15.0, 0.0, 15.0, 17.0, 11.0, 13.0, 13.0]
print("total overlap weight:", sum(solver_edge_weights))
print("violations (zero-weight steps):", solver_edge_weights.count(0.0))

total overlap weight: 269.0
violations (zero-weight steps): 4


In [5]:
# Map each path back to the original read ids and score it by total overlap weight.
# This is the step that makes a solved path usable: the reads file is indexed by read id,
# so a path still in relabelled ids would pull the wrong sequences.
#
# Stage 2 held reduced_graph / new_to_original / paths in memory. This notebook runs on
# its own, so all three are rebuilt here from the alignment records read above: the graph
# is the same one (this example has no small components to drop), and the relabelling
# follows networkx insertion order exactly as Stage 2 does.
reduced_graph = nx.DiGraph()
for i in range(len(scores)):
    if scores[i] > 0:
        reduced_graph.add_edge(int(edge_source[i]), int(edge_target[i]), weight=scores[i])

original_to_new = {original: new for new, original in enumerate(reduced_graph.nodes())}
new_to_original = {new: original for original, new in original_to_new.items()}

paths = [solver_path]    # the solved order(s) to score, in relabelled ids

MIN_PATH_WEIGHT = 50     # paths below this total overlap are not carried forward

paths_original_ids = []
path_weights = []
kept_path_indices = []

for j, path in enumerate(paths):
    original_vertex = [new_to_original[v] for v in path]
    print(original_vertex)
    paths_original_ids.append(original_vertex[:])

    total_weight = 0
    for i in range(len(original_vertex) - 1):
        edge_data = reduced_graph.get_edge_data(original_vertex[i], original_vertex[i + 1])
        total_weight += edge_data['weight'] if edge_data and 'weight' in edge_data else 0

    if total_weight >= MIN_PATH_WEIGHT:
        path_weights.append(total_weight)
        kept_path_indices.append(j)
        print(j, total_weight)

[3, 5, 10, 11, 23, 2, 8, 13, 17, 19, 9, 14, 15, 22, 0, 1, 7, 20, 4, 6, 12, 16, 18, 21]
0 269.0


In [6]:
# Get the read sequence for each node of the chosen path, from the reads file written by
# 1_multithread_alignment.py (one retained read per line, so line number = read id).
#
# -------------------------------------------------------------------------------
# CHECK THE ID SPACE. node_list_end below is the path in RELABELLED solver ids, while
# the reads file is indexed by ORIGINAL read ids. For this graph the equivalent
# original-id order is paths_original_ids[0], printed by the previous cell. The two are
# different permutations, so they build different contigs:
#     relabelled order  -> 92,819 bp contig, 1,518 bases of overlap collapsed
#     original-id order -> 92,701 bp contig, 1,636 bases of overlap collapsed
# The original-id order collapses more, as expected of the correct pairing, though the
# margin is small because the overlaps are short either way. Prefer
# paths_original_ids[0] unless your reads file is itself written in relabelled order.
# -------------------------------------------------------------------------------
node_list_end=[0,2,8,11,14,21,6,22,19,17,23,7,16,20,3,4,5,12,15,13,18,1,10,9]
#node_list_end=[0,6,17,23,7,16,20,13,10,2,8,11,14,22,19,3,4,15,18,21,5,12,1,9]
#node_list_end= Sequence1
#node_list_end=[0,11,6,16,15,18,2,22,19,20,3,4,13,1,10,8,14,21,17,23,7,5,12,9]
#node_list_end=[0,8,14,21,6,16,20,4,13,2,22,19,17,23,12,15,1,11,7,3,5,18,10,9]
#node_list_end=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
specific_list=node_list_end
# for i in range(len(specific_list)):
#     print(node_i[specific_list[i]], node_j[specific_list[i]], score[specific_list[i]])
sequences=[]
with open("cleaned_reads.txt", 'r') as file:
    lines=file.readlines()
    for line in lines:
        sequences.append(line.strip())

for i in range(len(specific_list)):
    print(sequences[specific_list[i]])

#print(len(score))

...


In [7]:
# Generate the contig from the ordered reads (Supplementary Section S1 E, Eqs. S5-S6).
#
# Starting from the first read of the path, each subsequent read is appended after the
# region it shares with the growing contig has been removed:
#     overlap = aligned columns with a gap in NEITHER sequence, i.e. matches + mismatches   (Eq. S5)
#     contig  = contig + next_read[overlap:]                                                (Eq. S6)
#
# Only the terminal 100 bases of each side are aligned: the junction between consecutive
# reads is a suffix-to-prefix overlap, so aligning the full reads would be slower and
# liable to match an internal repeat instead of the true junction.
#
# Where the solved path carries a violation the two reads do not really overlap, the local
# alignment finds little, and nearly the whole read is appended - which is why a violating
# path yields a longer, more redundant contig and lower BLASTn query coverage.
#
# pairwise2 is used here rather than Bio.Align.PairwiseAligner on purpose: where several
# alignments share the best score the two back-ends return different tracebacks, and the
# overlap length - hence the contig - depends on which one is returned.

from Bio import pairwise2
# from Bio.pairwise2 import format_alignment  # optional debug

# contig: growing joined sequence
contig = sequences[specific_list[0]]
overlap_list = []

for i in range(1, len(specific_list)):
    next_seq = sequences[specific_list[i]]

    contig_window = contig[-100:] if len(contig) >= 100 else contig
    next_window = next_seq[:100] if len(next_seq) >= 100 else next_seq

    alignments = pairwise2.align.localms(contig_window, next_window, 1, -1, -1, -1)
    aligned_contig_part, aligned_next_part, score, start, end = alignments[0]

    matches = mismatches = gaps = 0
    for a, b in zip(aligned_contig_part, aligned_next_part):
        if a == '-' or b == '-':
            gaps += 1
        elif a == b:
            matches += 1
        else:
            mismatches += 1

    overlap = matches + mismatches   # the actual overlap length

    # recorded so every junction can be inspected afterwards, as
    # (overlap length, position along the path, read id)
    overlap_list.append((overlap, i, specific_list[i]))

    # build contig
    contig = contig + next_seq[overlap:]

max_overlap, max_i, max_seq_id = max(overlap_list, key=lambda x: x[0])

# The widest junction found. A high value means two reads genuinely abutted; junctions
# near zero are the ones that contributed no overlap and simply concatenated.
print("Highest overlap:", max_overlap)
print("Occurs at loop index:", max_i)
print("Sequence ID:", max_seq_id)
print("Sequence:", sequences[max_seq_id])

Highest overlap: 92
Occurs at loop index: 17
Sequence ID: 12
Sequence: CGATGTACTTCGTTCAGTTACGTATTGCTAAGGTTAAAAGTTCATTCCCACGGTAACACCAGCACGCGCGCACTTTTTTTTTTTTTCGCCTATGGAGAAGGGTGCCATGACAAGATATTTTTTACATGAACATTGAAAACAGTCTTGTGGTGGACTTTAGACTCTTTTTTCCTTTATGTTGAATATACTCATTATCCCAGAATCCTTCCTTTCACGAAAGAGTCACAGCTTGTTTTTTGTCTTTCTAGGGCCACACCTGTGGCATATGGAGTTCAGGCTAGGGATCTAATCACAGCTGTAGCCGCTGGCCTTTGCCAAGCCTCCCAGCAATGCCAAATCTGAGCTGTGTCTGCGACCTACACCACAGCTCATGGCAACGCCGGATCCCTTAACCCACTGAGCAGGCCAGGCATCGGACCTGCAACCTCATGGTTCCTAGTTGAATTTATTTAACCACTGAGCCACGATGGGAACTCCAAAGTCACAACTTCTTGCATCGGGATTGCATGCTCCTTCTACTTTTCATTTATGGCGGTCCAGAGTCTCTTCCAGGGATCTTCTCATCAAGAATCTCTGAGGCGCGACACTACTCCTGCCCCTGGAGACTGATGTCAATGGGCTATACAAATTAGACTTTACTGACATCTGATTTCCAGTTGATTTCAACCAGTGAGAGGGAACCAACAGGTGATCAGTGTTGAGAAACAAGAGAAGTTAGAATGAATTCTCCATCTGCTCCCTCCAGCCCAGGCTGAAATTTGCTAGTGGCTTTGTTTTCCCTATTGAAAGTCAAGTTCAAAGAGACAGATAATAAATGGGTATTTTCCAGCAACTTGGGGAAGAGGAAAATGGGGAGTGATTGCTAATGGATATGTGGCTTTAGCAGCTGATTAAAATATTCTTGAATTGGTGATAGTTGCAAAACTTTT

#with this going and checking for the organism in the BLASTN